# RLCFrictionLM — Deep Training Run
### Full L+R+C circuit neurons: watching physics emerge across layers

Each neuron is a damped harmonic oscillator with **learned** inductance (L),
resistance (R), and capacitance (C). Starting from identical critical damping,
the network freely discovers which layers should resonate and which should stabilise.

**What to watch during training:**
- `ω₀ spread` — natural frequencies diverging across layers (starts at 0, grows)
- `underdamped layers` — how many layers went resonant (ζ < 1)
- `sparsity` — fraction of neurons silent (target: 70%+ for CPU advantage)


## 1 · Clone repo & install

In [ ]:
!pip install tiktoken datasets --quiet

import os, subprocess, sys

if not os.path.exists("frictionLLM"):
    subprocess.run(["git", "clone",
                    "https://github.com/yossibello/frictionLLM.git"], check=True)
    print("Cloned frictionLLM")
else:
    subprocess.run(["git", "-C", "frictionLLM", "pull"], check=True)
    print("Updated frictionLLM")

sys.path.insert(0, "frictionLLM")
os.chdir("frictionLLM")
print("Working dir:", os.getcwd())


## 2 · GPU setup

In [ ]:
import os, math, time, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = (
    torch.device("cuda") if torch.cuda.is_available() else
    torch.device("mps")  if torch.backends.mps.is_available() else
    torch.device("cpu")
)
n_gpus = torch.cuda.device_count() if device.type == "cuda" else 0
print(f"Device : {device}")
if device.type == "cuda":
    for i in range(n_gpus):
        p = torch.cuda.get_device_properties(i)
        print(f"  GPU {i}: {p.name}  {p.total_memory/1e9:.1f} GB")
    print(f"GPUs   : {n_gpus}  ({'DataParallel' if n_gpus > 1 else 'single GPU'})")


## 3 · Verify imports

In [ ]:
from friction_llm import (
    FrictionConfig, RLCFrictionLM, SharpnessCurriculum, RLCNeuron
)
cfg_test = FrictionConfig.tiny()
cfg_test.use_rlc = True
m = RLCFrictionLM(cfg_test)
print(f"Import OK — tiny test model: {m.param_count()/1e6:.1f}M params")
del m, cfg_test


## 4 · Data — WikiText-103

In [ ]:
import os
os.makedirs("data", exist_ok=True)

try:
    from datasets import load_dataset
    print("Loading WikiText-103 (~103M tokens)...")
    ds = load_dataset("wikitext", "wikitext-103-raw-v1")
    with open("data/input.txt", "w", encoding="utf-8") as f:
        for split in ["train", "validation", "test"]:
            for row in ds[split]:
                text = row["text"].strip()
                if text:
                    f.write(text + "\n")
    print(f"Saved: {os.path.getsize('data/input.txt')/1e6:.0f} MB")
except Exception as e:
    import urllib.request
    print(f"Falling back to TinyShakespeare ({e})")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
        "data/input.txt"
    )
    print(f"Downloaded: {os.path.getsize('data/input.txt')/1e6:.1f} MB")


In [ ]:
import tiktoken, numpy as np

enc = tiktoken.get_encoding("gpt2")
with open("data/input.txt", encoding="utf-8") as f:
    text = f.read()

tokens = np.array(enc.encode_ordinary(text), dtype=np.uint16)
split  = int(0.9 * len(tokens))
tokens[:split].tofile("data/train.bin")
tokens[split:].tofile("data/val.bin")
print(f"Train : {split:,} tokens")
print(f"Val   : {len(tokens)-split:,} tokens")


In [ ]:
class TokenDataset:
    def __init__(self, path, seq_len):
        self.data    = np.memmap(path, dtype=np.uint16, mode="r")
        self.seq_len = seq_len

    def __len__(self):
        return max(0, len(self.data) - self.seq_len - 1)

    def get_batch(self, batch_size, device):
        ix = torch.randint(len(self), (batch_size,))
        x  = torch.stack([torch.from_numpy(
                self.data[i : i+self.seq_len].astype(np.int64)) for i in ix])
        y  = torch.stack([torch.from_numpy(
                self.data[i+1 : i+1+self.seq_len].astype(np.int64)) for i in ix])
        return x.to(device), y.to(device)

SEQ_LEN  = 512
train_ds = TokenDataset("data/train.bin", SEQ_LEN)
val_ds   = TokenDataset("data/val.bin",   SEQ_LEN)
print(f"Seq len : {SEQ_LEN}")
print(f"Train   : {len(train_ds):,} positions")
print(f"Val     : {len(val_ds):,} positions")


## 5 · Training function

Logs ω₀ spread and underdamped layer count every step — watch the physics emerge.

In [ ]:
from friction_llm import FrictionConfig, RLCFrictionLM, SharpnessCurriculum

def unwrap(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def circuit_snapshot(base):
    """Per-layer ω₀, ζ, underdamped %, sparsity — for live monitoring."""
    rows = []
    for block in base.blocks:
        rlc  = block.rlc_block.rlc
        fric = block.rlc_block.friction
        rows.append({
            "omega_0": rlc.omega_0.mean().item(),
            "zeta":    rlc.damping_ratio.mean().item(),
            "underdamped_pct": (rlc.damping_ratio < 1.0).float().mean().item() * 100,
            "mu_s":    fric.mu_s.mean().item(),
        })
    return rows

def train_rlc(model, train_ds, val_ds,
              max_steps=10000, batch_size=16,
              lr=3e-4, log_every=100,
              ckpt_every=1000, ckpt_dir="checkpoints"):

    os.makedirs(ckpt_dir, exist_ok=True)
    use_amp = device.type == "cuda"
    scaler  = torch.amp.GradScaler("cuda", enabled=use_amp)

    effective_batch = batch_size * max(n_gpus, 1)
    if n_gpus > 1:
        model = nn.DataParallel(model)
        print(f"DataParallel: {n_gpus} GPUs  effective batch={effective_batch}")

    base = unwrap(model)

    physics, other = [], []
    for n, p in base.named_parameters():
        if any(k in n for k in ("raw_mu","raw_ratio","log_L","log_R","log_C")):
            physics.append(p)
        else:
            other.append(p)

    optimizer = torch.optim.AdamW(
        [{"params": other,   "weight_decay": 0.1},
         {"params": physics, "weight_decay": 0.0}],
        lr=lr, betas=(0.9, 0.95)
    )

    curriculum = SharpnessCurriculum(base, base.config)

    history = {
        "step": [], "loss": [], "val_loss": [], "sparsity": [],
        "omega_spread": [], "zeta_spread": [],
        "underdamped_layers": [], "sharpness": [],
        "layers": [],   # full per-layer snapshot at each log step
    }
    lr_min = lr / 10
    t0     = time.time()

    for step in range(max_steps + 1):
        # Cosine LR with warmup
        warmup = min(500, max_steps // 10)
        if step < warmup:
            cur_lr = lr * step / max(warmup, 1)
        else:
            progress = (step - warmup) / (max_steps - warmup)
            cur_lr = lr_min + 0.5 * (lr - lr_min) * (1 + math.cos(math.pi * progress))
        for pg in optimizer.param_groups:
            pg["lr"] = cur_lr

        model.train()
        x, y = train_ds.get_batch(effective_batch, device)
        with torch.amp.autocast("cuda", enabled=use_amp):
            _, loss = model(x, y)
        if loss.dim() > 0:
            loss = loss.mean()

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(base.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        sharpness = curriculum.step()

        # ── Log ──────────────────────────────────────────────────────────────
        if step % log_every == 0:
            model.eval()
            with torch.no_grad():
                xv, yv = val_ds.get_batch(effective_batch, device)
                _, vloss = model(xv, yv)
                if vloss.dim() > 0:
                    vloss = vloss.mean()

            # Sparsity + circuit snapshot (use base model directly, no DataParallel)
            sample, _ = val_ds.get_batch(4, device)
            report  = base.circuit_report(sample)
            sparsity = report["overall_sparsity"]
            snap    = circuit_snapshot(base)
            model.train()

            omegas = [r["omega_0"] for r in snap]
            zetas  = [r["zeta"]    for r in snap]
            omega_spread = max(omegas) - min(omegas)
            zeta_spread  = max(zetas)  - min(zetas)
            underdamped  = sum(1 for z in zetas if z < 1.0)

            dt = (time.time() - t0) / max(step, 1)
            history["step"].append(step)
            history["loss"].append(loss.item())
            history["val_loss"].append(vloss.item())
            history["sparsity"].append(sparsity)
            history["omega_spread"].append(omega_spread)
            history["zeta_spread"].append(zeta_spread)
            history["underdamped_layers"].append(underdamped)
            history["sharpness"].append(sharpness)
            history["layers"].append(snap)

            print(
                f"step {step:5d} | loss {loss.item():.4f} | val {vloss.item():.4f} "
                f"| sparse {sparsity:.1%} | ω₀spread {omega_spread:.4f} "
                f"| underdamped {underdamped}/{len(snap)} "
                f"| sharp {sharpness:.1f} | {dt*1000:.0f}ms/step"
            )

        # ── Checkpoint ───────────────────────────────────────────────────────
        if step % ckpt_every == 0 and step > 0:
            path = f"{ckpt_dir}/rlc_step_{step:06d}.pt"
            torch.save({
                "model":      base.state_dict(),
                "optimizer":  optimizer.state_dict(),
                "curriculum": curriculum.state_dict(),
                "step":       step,
                "history":    history,
                "config":     base.config,
            }, path)
            print(f"  → saved {path}")

    return history, base


## 6 · Train — RLCFrictionLM (117M params, 10 000 steps)

Expected time on 2× T4: **~3 hours**.
Checkpoints saved every 1 000 steps → session can be resumed.

Key milestones to watch:
- **step ~500**: warmup ends, LR hits peak
- **step ~1000**: sharpness curriculum starts annealing
- **step ~2000**: ω₀ spread starts growing — layers finding different frequencies
- **step ~5000**: resonant vs overdamped pattern solidifies


In [ ]:
# Medium config: 117M params, 12 layers — right size for WikiText-103
cfg = FrictionConfig.medium()
cfg.max_seq_len = SEQ_LEN
cfg.use_rlc     = True
cfg.mu_s_init   = 0.05   # charge scale is ~dt²×V, much smaller than raw signal
cfg.rlc_dt      = 0.3    # larger step → more charge per layer

model = RLCFrictionLM(cfg).to(device)
print(f"RLCFrictionLM: {model.param_count()/1e6:.1f}M params")
print(f"Layers: {cfg.n_layers}  d_model: {cfg.d_model}  d_ff: {cfg.d_ff}")
print(f"μ_s={cfg.mu_s_init}  rlc_dt={cfg.rlc_dt}")
print()
print("Training for 10 000 steps — circuit physics will emerge after ~2 000")
print("Watch: ω₀ spread (frequency divergence) + underdamped layers (resonance)")

history, model = train_rlc(
    model, train_ds, val_ds,
    max_steps=10000,
    batch_size=16,
    lr=3e-4,
    log_every=100,
    ckpt_every=1000,
)


## 7 · Circuit physics report

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

sample_ids = torch.tensor(
    enc.encode_ordinary("The relationship between"),
    dtype=torch.long, device=device
).unsqueeze(0)

model.eval()
model.print_circuit_report(sample_ids)


## 8 · Visualise circuit evolution

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

steps = history["step"]
n_layers = len(history["layers"][0]) if history["layers"] else 0

fig = plt.figure(figsize=(18, 12))
fig.suptitle("RLCFrictionLM — Circuit Physics Emerging During Training",
             fontsize=14, fontweight="bold")
gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Loss ─────────────────────────────────────────────────────────────────────
ax = fig.add_subplot(gs[0, :2])
ax.plot(steps, history["loss"],     label="train", alpha=0.7, linewidth=1.5)
ax.plot(steps, history["val_loss"], label="val",   linewidth=2)
ax.set(xlabel="Step", ylabel="Loss", title="Training & Validation Loss")
ax.legend(); ax.grid(alpha=0.3)

# ── Sparsity ─────────────────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.plot(steps, [s*100 for s in history["sparsity"]], color="steelblue", linewidth=2)
ax2.axhline(70, linestyle="--", color="gray", alpha=0.5, label="CPU target (70%)")
ax2.set(xlabel="Step", ylabel="Sparsity %", title="Gate Sparsity Over Training")
ax2.legend(); ax2.grid(alpha=0.3)

# ── ω₀ spread (key metric: are layers finding different frequencies?) ─────────
ax3 = fig.add_subplot(gs[1, :2])
ax3.plot(steps, history["omega_spread"], color="darkorange", linewidth=2)
ax3.set(xlabel="Step", ylabel="max(ω₀) − min(ω₀)",
        title="ω₀ Spread Across Layers  (0 = all same,  > 0 = differentiated)")
ax3.grid(alpha=0.3)

# ── Underdamped layer count ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])
ax4.plot(steps, history["underdamped_layers"], color="crimson", linewidth=2,
         drawstyle="steps-post")
ax4.set(xlabel="Step", ylabel="Layers with ζ < 1",
        title=f"Resonant Layers  (out of {n_layers})")
ax4.set_yticks(range(n_layers + 1)); ax4.grid(alpha=0.3)

# ── Per-layer ω₀ evolution (heatmap over training) ───────────────────────────
ax5 = fig.add_subplot(gs[2, :2])
if history["layers"] and n_layers > 0:
    omega_matrix = np.array([[r["omega_0"] for r in snap]
                              for snap in history["layers"]]).T   # [n_layers, steps]
    im = ax5.imshow(omega_matrix, aspect="auto", cmap="RdYlGn",
                    extent=[steps[0], steps[-1], n_layers-0.5, -0.5])
    plt.colorbar(im, ax=ax5, label="ω₀")
    ax5.set(xlabel="Step", ylabel="Layer", yticks=range(n_layers),
            yticklabels=[f"L{i}" for i in range(n_layers)],
            title="ω₀ per Layer Over Training  (green=higher freq, red=lower freq)")

# ── Per-layer ζ at end of training ───────────────────────────────────────────
ax6 = fig.add_subplot(gs[2, 2])
if history["layers"]:
    final_zeta = [r["zeta"] for r in history["layers"][-1]]
    colors = ["crimson" if z < 1.0 else "steelblue" for z in final_zeta]
    ax6.barh(range(n_layers), final_zeta, color=colors)
    ax6.axvline(1.0, color="black", linestyle="--", linewidth=1.5, label="ζ=1 (critical)")
    ax6.set(xlabel="Damping ratio ζ", yticks=range(n_layers),
            yticklabels=[f"L{i}" for i in range(n_layers)],
            title="Final ζ per Layer
(red=resonant, blue=overdamped)")
    ax6.legend(fontsize=8); ax6.grid(alpha=0.3, axis="x")

plt.savefig("rlc_physics.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rlc_physics.png")


## 9 · Sparsity detail

In [ ]:
sample, _ = val_ds.get_batch(8, device)
model.eval()
model.print_circuit_report(sample)


## 10 · Text generation

In [ ]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")

prompts = [
    "The theory of",
    "In the year 1900",
    "Scientists discovered that",
]

for prompt in prompts:
    ids = enc.encode_ordinary(prompt)
    idx = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    out = model.generate(idx, max_new_tokens=120, temperature=0.8, top_k=40)
    print(f"Prompt: {prompt!r}")
    print(enc.decode(out[0].tolist()))
    print("-" * 60)


## Resuming from checkpoint

If the Kaggle session restarted, resume from the last checkpoint:

```python
ckpt = torch.load("checkpoints/rlc_step_005000.pt", map_location=device)
cfg  = ckpt["config"]
model = RLCFrictionLM(cfg).to(device)
model.load_state_dict(ckpt["model"])
history = ckpt["history"]
print(f"Resumed from step {ckpt['step']}, last loss {history['loss'][-1]:.4f}")
```

Then call `train_rlc(model, ...)` with `max_steps` set to whatever you want to continue to.
